# Plan D: CamoEdge, boundary-guided camouflaged object segmentation

Run the cells from top to bottom. Read `plans/04-camoedge.md` first. This task is plain PyTorch and does not use Ultralytics. Before you start:

1. Use **Runtime > Change runtime type** and pick a GPU (for example T4).
2. Get COD10K and CAMO from their authors' release pages (see `datasets/manifest.json`), arrange them as below, zip the folder as `camo.zip` and upload it to the top level of Google Drive. Check each README for the licence, and keep the copy private.
3. Edit the **Configuration** cell: the experiment, the seed and the number of epochs.

Expected dataset layout inside the zip (training set = COD10K-Tr + CAMO-Tr, 4,040 images; NC4K is optional):

```
camo/
  TrainDataset/Imgs    TrainDataset/GT
  TestDataset/COD10K/Imgs   TestDataset/COD10K/GT
  TestDataset/CAMO/Imgs     TestDataset/CAMO/GT
  TestDataset/NC4K/Imgs     TestDataset/NC4K/GT
```

## 1. Install and check the GPU

In [ ]:
# Clone the project (or update it when the runtime is reused), install the pinned requirements and show the GPU
import os
import subprocess
import sys
from pathlib import Path
# Address of the team repository
repositoryUrl = "https://github.com/keithku321/object-detection"
# Folder on the Colab machine that holds the clone
repositoryFolder = Path("/content/objectDetection")
# A private repository needs a GitHub token stored as the Colab Secret GITHUB_TOKEN (key icon in the left bar); never paste a token into a cell
try:
    from google.colab import userdata
    githubToken = userdata.get("GITHUB_TOKEN")
except Exception:
    githubToken = None
cloneUrl = repositoryUrl.replace("https://", f"https://{githubToken}@") if githubToken else repositoryUrl
gitCommand = ["git", "-C", str(repositoryFolder), "pull", "--quiet"] if repositoryFolder.exists() else ["git", "clone", "--quiet", cloneUrl, str(repositoryFolder)]
gitResult = subprocess.run(gitCommand, capture_output=True, text=True)
# Print git's message with the token removed, so the token can never end up in a saved notebook
if gitResult.returncode != 0: print(gitResult.stderr.replace(githubToken or "GITHUB_TOKEN", "***"))
os.chdir(repositoryFolder)
if str(repositoryFolder) not in sys.path: sys.path.insert(0, str(repositoryFolder))
# Colab already has PyTorch; requirements.txt adds the pinned Ultralytics and the task libraries without touching it
!pip install -q -r requirements.txt
# Show the attached GPU; if this prints an error then change the runtime type to a GPU
!nvidia-smi
from common.environment import printEnvironment
printEnvironment()

## 2. Configuration

In [ ]:
# Name of this task; it names the results file and the run folders
taskName = "camoedge"
# Name of the zip file that you uploaded to the top level of your Google Drive
datasetZipName = "camo.zip"
# Name of the folder that appears after unzipping, which must match the folder inside the zip
datasetFolderName = "camo"
# Experiment to train in section 7: E2 (structure loss), E3 (edge branch), E4 (edge loss) or E5 (full CamoEdge); see camoedge/model.py
experimentName = "E5"
# Encoder from the timm library: pvt_v2_b2 or resnet50
encoderName = "pvt_v2_b2"
# Seed; exploration uses 0, the final runs use 0, 1 and 2
seed = 0
# Number of full passes over the training data
trainingEpochs = 30
# Side length in pixels that images are resized to
imageSize = 352
# Images per batch; lower it if the GPU runs out of memory
batchSize = 16
# Train on only this many images for a quick trial; None uses all 4,040 training images
subsetCount = None
# Drive folder for results, so that weights survive when Colab resets the runtime
driveResultsFolder = Path("/content/drive/MyDrive/yoloProjectResults")
# Local copy of the zip file on the Colab machine
localZipPath = Path("/content") / datasetZipName
# Local folder that will hold the unzipped dataset
datasetRootPath = Path("/content/datasets") / datasetFolderName

## 3. Mount Google Drive and unpack the dataset

The zip is copied to local storage first because training directly from Drive is slow. Its SHA-256 is compared with `datasets/manifest.json`; when no checksum is recorded yet, the real one is printed so you can add it.

In [ ]:
from google.colab import drive
# Mount Google Drive; a pop-up will ask for permission. Drive holds results and checkpoints only, never the working copy of the dataset
drive.mount("/content/drive")
driveResultsFolder.mkdir(parents=True, exist_ok=True)
# Bring back results rows of earlier sessions: a fresh clone only contains the rows that were committed
from common.results import syncResultFiles
print(syncResultFiles(taskName, driveResultsFolder / "results"), "results rows so far")

from common.datasets import extractZip, getManifestEntry, verifySha256
# Copy the zip to local storage because training straight from Drive is slow
!cp "/content/drive/MyDrive/{datasetZipName}" "{localZipPath}"
if verifySha256(localZipPath, getManifestEntry("camoedge").get("sha256")) is False: raise ValueError("The zip does not match the checksum in datasets/manifest.json")
extractZip(localZipPath, "/content/datasets")

## 4. Helper functions

The logic lives in `common/` and `camoedge/` so it can be tested on a PC; this cell only imports it and adds two display helpers.

In [ ]:
from IPython.display import display
from PIL import Image
# File endings that count as images
imageSuffixes = {".jpg", ".jpeg", ".png"}

# Display the first few images from a folder inside the notebook
def showSavedImages(folderPath, maximumCount=4):
    imageFiles = [file for file in sorted(Path(folderPath).iterdir()) if file.suffix.lower() in imageSuffixes]
    for imageFile in imageFiles[:maximumCount]:
        display(Image.open(imageFile))

# Save the results rows to Drive and print the summary table (mean ± std over seeds)
def saveAndShowResults():
    from common.results import syncResultFiles
    from common.summariseResults import writeSummary
    syncResultFiles(taskName, driveResultsFolder / "results")
    print(writeSummary() or "No results rows yet.")

import numpy as np
from camoedge.dataset import CamouflageDataset, collectTestPairs, collectTrainingPairs, superClassOf
from camoedge.evaluate import evaluateAndLog, loadCheckpointModel
from camoedge.train import exportFinalWeights, trainCamoEdge

## 5. Sanity-check the data

There is no dataset.yaml for this task; the loader pairs every image with the mask of the same name. Check the counts against the protocol (4,040 training images; 2,026 COD10K and 250 CAMO test images) and look at one image with its mask and the edge ground truth derived from the mask.

Read any warnings carefully. Most training problems are really data problems.

In [ ]:
trainingPairs = collectTrainingPairs(datasetRootPath)
testPairsByName = collectTestPairs(datasetRootPath)
print(len(trainingPairs), "training pairs;", {testName: len(pairs) for testName, pairs in testPairsByName.items()})
# The per-category analysis needs the super-class in the COD10K file names; "unknown" here means the naming pattern is different
print("Super-classes found:", sorted({superClassOf(imagePath.name) for imagePath, _ in testPairsByName.get("COD10K", [])}))
# Show one training image, its mask and its edge ground truth side by side
sample = CamouflageDataset(trainingPairs, imageSize)[0]
sampleImage = Image.open(trainingPairs[0][0]).convert("RGB").resize((imageSize, imageSize))
sampleMask, sampleEdge = (np.repeat(sample[key][0].numpy()[:, :, None] * 255, 3, axis=2).astype(np.uint8) for key in ("mask", "edge"))
display(Image.fromarray(np.concatenate([np.asarray(sampleImage), sampleMask, sampleEdge], axis=1)))

## 6. Baseline

Train the plain encoder and decoder with BCE only (experiment E1). It gives a working model, a data loader and metrics that are known to be right before any edge module is added. A finished baseline is skipped.

Compare with a published reference (experiment E0, for example SINet-V2) by quoting its numbers with a citation; do not claim state of the art.

In [ ]:
# Train E1 with a checkpoint on Drive after every epoch, then score it on the test sets and log one row per test set
baselineCheckpoint = trainCamoEdge("E1", datasetRootPath, driveResultsFolder, seed, trainingEpochs, batchSize, imageSize, encoderName=encoderName, maximumCount=subsetCount)
evaluateAndLog(baselineCheckpoint, datasetRootPath, notes=f"subset={subsetCount}" if subsetCount else "")
saveAndShowResults()

## 7. Train

Train the experiment chosen in the configuration cell. A checkpoint is written to Drive after every epoch so the run survives a disconnect. The time per epoch is printed: use it for the compute budget (gate G1).

The reported model is the one after the final epoch. The standard protocol has no validation split, and choosing the best epoch on the test sets would be model selection on test data.

In [ ]:
# Train the chosen experiment, score it on the test sets and log one row per test set in results/camoedge.csv
experimentCheckpoint = trainCamoEdge(experimentName, datasetRootPath, driveResultsFolder, seed, trainingEpochs, batchSize, imageSize, encoderName=encoderName, maximumCount=subsetCount)
evaluateAndLog(experimentCheckpoint, datasetRootPath, notes=f"subset={subsetCount}" if subsetCount else "")
saveAndShowResults()

### If Colab disconnected during training

Run the earlier setup cells (1 to 5) again, then run the cell below. It is the same call as the training cell: every run has a fixed folder on Drive, so training continues from the last saved epoch, and finished runs are skipped.

In [ ]:
# Continue the interrupted run from its last checkpoint on Drive (same call as above; finished runs are skipped)
# experimentCheckpoint = trainCamoEdge(experimentName, datasetRootPath, driveResultsFolder, seed, trainingEpochs, batchSize, imageSize, encoderName=encoderName, maximumCount=subsetCount)
# evaluateAndLog(experimentCheckpoint, datasetRootPath, notes=f"subset={subsetCount}" if subsetCount else "")
# saveAndShowResults()

## 8. Evaluate the trained model

Accuracy rows are already logged: S-measure, E-measure, weighted F-measure, MAE and the boundary F-score. This cell adds model complexity and speed (experiment E9). The project has no Jetson, so edge-hardware claims rest on these numbers.

In [ ]:
import time
import torch
from camoedge.model import countParameters
# Parameters and frames per second of the trained model at the training image size (batch 1, FP16 on the GPU)
segmenter, segmenterSettings = loadCheckpointModel(experimentCheckpoint)
device = next(segmenter.parameters()).device
dummyInput = torch.rand(1, 3, imageSize, imageSize, device=device)
with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
    for _ in range(50): segmenter(dummyInput)
    if device.type == "cuda": torch.cuda.synchronize()
    startTime = time.perf_counter()
    for _ in range(300): segmenter(dummyInput)
    if device.type == "cuda": torch.cuda.synchronize()
framesPerSecond = 300 / (time.perf_counter() - startTime)
print(f"{segmenterSettings['variantName']}: {countParameters(segmenter):,} parameters, {framesPerSecond:.1f} FPS on {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'}")

## 9. Look at real predictions

Scores hide failures, so look at the images: the mask, the edge map, and where the boundary is wrong.

In [ ]:
from camoedge.demoTab import makeOverlay
from camoedge.evaluate import predictProbability
from camoedge.dataset import preprocessImage
# Show image, predicted mask and overlay for a few test images
for imagePath, maskPath in testPairsByName["COD10K"][:4]:
    testImage = np.asarray(Image.open(imagePath).convert("RGB"))
    probability = predictProbability(segmenter, preprocessImage(testImage, imageSize).to(device), testImage.shape[:2])
    predictedMask = np.repeat((probability >= 0.5)[:, :, None] * 255, 3, axis=2).astype(np.uint8)
    display(Image.fromarray(np.concatenate([testImage, predictedMask, makeOverlay(testImage, probability >= 0.5)], axis=1)).resize((900, 300)))

## 10. Per-category and boundary analysis

This is the cell that turns the project into a finding. Global scores hide boundary gains because edge pixels are under 5% of a mask, so compare the **boundary F-score** between the baseline and the edge-guided model, and look at the COD10K super-classes separately.

In [ ]:
import csv
from common.results import readResultRows, resultsFolder
# Boundary F-score and the standard metrics for every experiment logged so far
for row in readResultRows(resultsFolder / "camoedge.csv"):
    print(f"{row['variant']:<22} seed {row['seed']}  S = {float(row['sAlpha']):.3f}  E = {float(row['eMean']):.3f}  Fw = {float(row['fWeighted']):.3f}  MAE = {float(row['mae']):.3f}  boundaryF = {float(row['boundaryF']):.3f}")
# Per-super-class table of the experiment trained above (written next to its checkpoint)
categoryTablePath = Path(experimentCheckpoint).parent / "perCategory.csv"
if categoryTablePath.exists():
    with categoryTablePath.open(newline="") as csvFile:
        for categoryRow in csv.DictReader(csvFile): print({key: (value if key == "superClass" else round(float(value), 3)) for key, value in categoryRow.items()})

## 11. Export and download (optional)

Save a slim copy of the weights for the demo app (`weights/` folder of the repository).
The results rows now sit in `results/camoedge.csv` inside the clone and in `yoloProjectResults/results/` on Drive. Rows are part of the repository: download the CSV (or copy it from Drive) and commit it on your branch, without weights. Strip the outputs of this notebook before committing it (**Edit > Clear all outputs**).

In [ ]:
from google.colab import files
# Save weights and settings only (no optimiser state) under the name <task>-<variant>-seed<k>.pt, then download the file
finalWeightsPath = exportFinalWeights(experimentCheckpoint, driveResultsFolder / "weights")
files.download(str(finalWeightsPath))
saveAndShowResults()